In [1]:
import pandas as pd
import eikon as ek
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import time
import openpyxl
from openpyxl import load_workbook

In [7]:

# today's date
today = pd.to_datetime(datetime.today().strftime('%Y-%m-%d'))

ice_report = 'ICE_082224'

# Load the Excel file
file_path = f'C:/Users/wangwz/OneDrive - Aramco Overseas Company/Trading/ICE_Reports/{ice_report}.xlsx'
df_ice = pd.read_excel(file_path, header=None)

## Preprocessing
cleared_deals_index = df_ice[df_ice.apply(lambda row: row.astype(str).str.contains('Cleared Deals').any(), axis=1)].index[0]
# Remove all rows before the "Cleared Deals" section
df_cleared = df_ice.iloc[cleared_deals_index+2:].reset_index(drop=True)
# Find the row index where the headers start (where "Trade Date" appears)
header_row_index = df_cleared[df_cleared.apply(lambda row: row.astype(str).str.contains('Trade Date').any(), axis=1)].index[0]
# Set the correct header
df_cleared.columns = df_cleared.iloc[header_row_index]
# Remove the header row from the data
df_cleared = df_cleared.iloc[header_row_index + 1:].reset_index(drop=True)


df_nap = df_cleared[df_cleared['Clearing Acct'] == '7B814']
print(df_nap)

0        Trade Date       Trade Time  Deal ID   Leg ID  Orig ID     B/S  \
18  August 22, 2024  02:38:40 PM GMT  3420585  3420587  3420583  Bought   
19  August 22, 2024  02:38:40 PM GMT  3420585  3420586  3420583    Sold   
20  August 22, 2024  02:54:19 PM GMT  3555215  3555217  3555213  Bought   
21  August 22, 2024  02:54:19 PM GMT  3555215  3555216  3555213    Sold   
22  August 22, 2024  02:57:59 PM GMT  3686038  3686040  3686036  Bought   
23  August 22, 2024  02:57:59 PM GMT  3686038  3686039  3686036    Sold   
26  August 22, 2024  03:19:42 PM GMT  3781791      NaN  3781789  Bought   
29  August 22, 2024  03:53:51 PM GMT  4083170      NaN  4083169    Sold   

0                 Product                                Hub Contract  \
18        Naphtha Futures                 Naphtha CIF NWE Cg   Sep24    
19        Naphtha Futures                 Naphtha CIF NWE Cg   Oct24    
20        Naphtha Futures                 Naphtha CIF NWE Cg   Sep24    
21        Naphtha Futures       

In [8]:
expo_columns = ['1640', 'Pricing Completion 1', 'Pricing Completion 2', 'Reference', 'Hub', 'Notes', 'Trade Date', 'SAPTrade#', 'Trade Book', 'TRADE CLASS', 'PRODUCT TYPE', 'TRADE TYPE', 'PRODUCT', 'TENOR','BUY/SELL', 'SIZE', 'UNIT', 'BBL/MT', 'PRICE']
df = pd.DataFrame(columns=expo_columns).fillna('')

df['Trade Date'] = pd.to_datetime(df_nap['Trade Date'])

df['Trade Book'] = 'NAPHTHA'

df['TRADE TYPE'] = 'FLATPRICE'

df['TENOR'] = pd.to_datetime(df_nap['Begin Date'], format='%d/%m/%Y')

df['BUY/SELL'] = np.where(df_nap['B/S'] == 'Bought', 'BUY', 'SELL')

df['SIZE'] = df_nap['Total Quantity']

df['UNIT'] = np.where(df_nap['Qty Units']== 'bbl', 'BBL', 'MT')

df['PRICE'] = df_nap['Price']

df['PRODUCT TYPE'] = np.where(df_nap['Product'].str.contains('Naphtha'), 'NAPHTHA', '')
df['PRODUCT TYPE'] = np.where(df_nap['Product'].str.contains('Freight'), 'NAPHTHA', df['PRODUCT TYPE'])
df['PRODUCT TYPE'] = np.where(df_nap['Product'].str.contains('Gasoline'), 'GASOLINE', df['PRODUCT TYPE'])
df['PRODUCT TYPE'] = np.where(df_nap['Product'].str.contains('Crude'), 'CRUDE', df['PRODUCT TYPE'])

df['PRODUCT'] = np.where(df_nap['Hub'].str.contains('Naphtha CIF'), 'NWE_Naphtha', '')
df['PRODUCT'] = np.where(df_nap['Hub'].str.contains('Naphtha C&F'), 'MOPJ_Naphtha', df['PRODUCT'])
df['PRODUCT'] = np.where(df_nap['Product'].str.contains('Gasoline'), 'EuroBOB', df['PRODUCT'])
df['PRODUCT'] = np.where(df_nap['Product'].str.contains('Crude'), 'BRENT_SWAP', df['PRODUCT'])
df['PRODUCT'] = np.where(df_nap['Hub'].str.contains('TC2'), 'TC2', df['PRODUCT'])
df['PRODUCT'] = np.where(df_nap['Hub'].str.contains('TC5'), 'TC5', df['PRODUCT'])
df['PRODUCT'] = np.where(df_nap['Hub'].str.contains('TC6'), 'TC6', df['PRODUCT'])
df['PRODUCT'] = np.where(df_nap['Hub'].str.contains('TC14'), 'TC14', df['PRODUCT'])

df['BBL/MT'] = np.where(df['PRODUCT TYPE'] == 'NAPHTHA', 8.9, 0)
df['BBL/MT'] = np.where(df['PRODUCT TYPE'] == 'GASOLINE', 8.33, df['BBL/MT'])
df['BBL/MT'] = np.where(df['PRODUCT TYPE'] == 'CRUDE', 7.33, df['BBL/MT'])

df['Reference'] = np.where(df_nap['Product'].str.contains('Naphtha Crack'), 'Nap Crk', '')
df['Reference'] = np.where(df_nap['Product'].str.contains('Gasoline Crack'), 'Gas Crk', df['Reference'])
df['Reference'] = np.where(df_nap['Product'].str.contains('Gasoline Diff Futures'), 'GASNAP', df['Reference'])
df['Reference'] = np.where(df_nap['Product'].str.contains('Naphtha Diff Futures'), 'EW', df['Reference'])
df['Reference'] = np.where(df_nap['Product'].str.contains('Crude Futures'), 'Brent', df['Reference'])
df['Reference'] = np.where(df_nap['Product'].str.contains('Naphtha Futures'), 'NAP FP', df['Reference'])
df['Reference'] = np.where(df_nap['Product'].str.contains('TC2'), 'TC2', df['Reference'])
df['Reference'] = np.where(df_nap['Product'].str.contains('TC5'), 'TC5', df['Reference'])
df['Reference'] = np.where(df_nap['Product'].str.contains('TC6'), 'TC6', df['Reference'])
df['Reference'] = np.where(df_nap['Product'].str.contains('TC14'), 'TC14', df['Reference'])

df['SIZE'] = np.where((df['UNIT'] == 'BBL') & (df['Reference'] == 'Nap Crk'), df['SIZE'] / 8.9, df['SIZE'])
df['SIZE'] = np.where((df['UNIT'] == 'BBL') & (df['Reference'] == 'Gas Crk'), df['SIZE'] / 8.33, df['SIZE'])
df['UNIT'] = np.where((df['UNIT'] == 'BBL') & (df['Reference'] == 'Nap Crk'), 'MT', df['UNIT'])
df['UNIT'] = np.where((df['UNIT'] == 'BBL') & (df['Reference'] == 'Gas Crk'), 'MT', df['UNIT'])

conditions_to_add = ['Nap Crk', 'Gas Crk', 'EW', 'GASNAP']

df_copy = df.copy()

new_rows = []

for i in range(len(df_copy)):
    if df_copy.iloc[i]['Reference'] in conditions_to_add:
        # Create a new row to insert
        new_row = df_copy.iloc[i].copy()
        new_row['Reference'] = '' 
        new_row['PRICE'] = '' 
        new_row['PRODUCT TYPE'] = ''
        new_row['PRODUCT'] = ''

        if df_copy.iloc[i]['Reference'] == 'Nap Crk':
            new_row['PRODUCT TYPE'] = 'CRUDE'
            new_row['PRODUCT'] = 'BRENT_SWAP'
            new_row['BBL/MT'] = 7.33
        elif df_copy.iloc[i]['Reference'] == 'Gas Crk':
            new_row['PRODUCT TYPE'] = 'CRUDE'
            new_row['PRODUCT'] = 'BRENT_SWAP'
            new_row['BBL/MT'] = 7.33
        elif df_copy.iloc[i]['Reference'] == 'EW':
            new_row['PRODUCT TYPE'] = 'NAPHTHA'
            new_row['PRODUCT'] = 'NWE_Naphtha'
        elif df_copy.iloc[i]['Reference'] == 'GASNAP':
            new_row['PRODUCT TYPE'] = 'NAPHTHA'
            new_row['PRODUCT'] = 'NWE_Naphtha'
            new_row['BBL/MT'] = 8.9
        
        new_rows.append((i, new_row))

# Insert new rows into the DataFrame
for pos, row in reversed(new_rows):
    df_copy = pd.concat([df_copy.iloc[:pos+1], pd.DataFrame([row], columns=df_copy.columns), df_copy.iloc[pos+1:]]).reset_index(drop=True)

print(df_copy)


  1640 Pricing Completion 1 Pricing Completion 2 Reference  Hub Notes  \
0  NaN                  NaN                  NaN    NAP FP  NaN   NaN   
1  NaN                  NaN                  NaN    NAP FP  NaN   NaN   
2  NaN                  NaN                  NaN    NAP FP  NaN   NaN   
3  NaN                  NaN                  NaN    NAP FP  NaN   NaN   
4  NaN                  NaN                  NaN    NAP FP  NaN   NaN   
5  NaN                  NaN                  NaN    NAP FP  NaN   NaN   
6  NaN                  NaN                  NaN   Nap Crk  NaN   NaN   
7  NaN                  NaN                  NaN            NaN   NaN   
8  NaN                  NaN                  NaN     Brent  NaN   NaN   

  Trade Date SAPTrade# Trade Book TRADE CLASS PRODUCT TYPE TRADE TYPE  \
0 2024-08-22       NaN    NAPHTHA         NaN      NAPHTHA  FLATPRICE   
1 2024-08-22       NaN    NAPHTHA         NaN      NAPHTHA  FLATPRICE   
2 2024-08-22       NaN    NAPHTHA         NaN     

In [9]:
excel_file_path = f'M:/24.Naphtha/Python scripts/Expo_inputs/ICE_{ice_report}.xlsx'
df_copy.to_excel(excel_file_path, index=False)

In [10]:
expo_data = pd.read_excel(excel_file_path)
print(expo_data)

   1640  Pricing Completion 1  Pricing Completion 2 Reference  Hub  Notes  \
0   NaN                   NaN                   NaN    NAP FP  NaN    NaN   
1   NaN                   NaN                   NaN    NAP FP  NaN    NaN   
2   NaN                   NaN                   NaN    NAP FP  NaN    NaN   
3   NaN                   NaN                   NaN    NAP FP  NaN    NaN   
4   NaN                   NaN                   NaN    NAP FP  NaN    NaN   
5   NaN                   NaN                   NaN    NAP FP  NaN    NaN   
6   NaN                   NaN                   NaN   Nap Crk  NaN    NaN   
7   NaN                   NaN                   NaN       NaN  NaN    NaN   
8   NaN                   NaN                   NaN     Brent  NaN    NaN   

  Trade Date  SAPTrade# Trade Book  TRADE CLASS PRODUCT TYPE TRADE TYPE  \
0 2024-08-22        NaN    NAPHTHA          NaN      NAPHTHA  FLATPRICE   
1 2024-08-22        NaN    NAPHTHA          NaN      NAPHTHA  FLATPRICE   
2 20